In [ ]:
!pip install -q langchain langchain-community langchain-huggingface langchain-text-splitters langchain-openai faiss-cpu pypdf chromadb

In [ ]:
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS, Chroma, Pinecone
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_openai import ChatOpenAI
from langchain_classic.chains import RetrievalQA
from langchain_community.document_loaders import PyPDFLoader, TextLoader, WebBaseLoader
from langchain_community.embeddings import OpenAIEmbeddings
import pinecone

**1. Document Loaders:**

In [ ]:
# Load PDF
loader = PyPDFLoader("document.pdf")
documents = loader.load()

In [ ]:
# Load from web
loader = WebBaseLoader("https://cs229.stanford.edu/main_notes.pdf")
documents = loader.load()

**2. Text Splitting:**

In [5]:
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200,
    length_function=len,
    separators=["\n\n", "\n", " ", ""]
)

chunks = text_splitter.split_documents(documents)

In [6]:
print(len(chunks))

4448


**3. Embeddings:**

In [ ]:
# OpenAI embeddings
embeddings = OpenAIEmbeddings()

In [8]:
# Hugging Face (free)
embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


**4. Vector Stores:**

In [ ]:
# FAISS (local)
vectorstore = FAISS.from_documents(chunks, embeddings)

In [ ]:
# Chroma (local with persistence)
vectorstore = Chroma.from_documents(
    chunks,
    embeddings,
    persist_directory="./chroma_db"
)

In [ ]:
# Pinecone (cloud)
pinecone.init(api_key="API_KEY", environment="us-east-1")
vectorstore = Pinecone.from_documents(chunks, embeddings, index_name="rag_index")

**5. Retrievers:**

In [ ]:
# Basic retriever
retriever = vectorstore.as_retriever(search_kwargs={"k":3})

In [ ]:
# Similarity search with score threshold
retriever = vectorstore.as_retriever(
    search_type='similarity_score_threshold',
    search_kwargs={"score_threshold": 0.7, "k": 3}
)

In [ ]:
# MMR (Maximum Marginal Relevance) - diversity
retriever = vectorstore.as_retriever(
    search_type="mmr",
    search_kwargs={"k": 3, "fetch_k": 10}
)